Import thư viện & Cấu hình

In [1]:
import tensorflow as tf
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K

# --- CẤU HÌNH ---
IMG_WIDTH, IMG_HEIGHT = 224, 224
BATCH_SIZE = 32
EPOCHS = 20        # Custom CNN nhẹ hơn nên có thể tăng Epochs lên 20-30
NUM_FOLDS = 5      # Số vòng lặp K-Fold
LEARNING_RATE = 0.001 # Tốc độ học (Custom CNN thường dùng 0.001, VGG hay dùng 0.0001)

# --- TỰ ĐỘNG PHÁT HIỆN MÔI TRƯỜNG ---
try:
    from google.colab import drive
    print("Detected: GOOGLE COLAB environment")
    drive.mount('/content/drive')
    # SỬA ĐƯỜNG DẪN NÀY THEO FOLDER TRÊN DRIVE CỦA BẠN
    DATA_DIR = '/content/drive/MyDrive/AI_VGG16_Classifier/data/raw' 
except ImportError:
    print("Detected: LOCAL environment")
    DATA_DIR = '../data/raw' 

print(f"Đang tìm dữ liệu tại: {DATA_DIR}")

Detected: LOCAL environment
Đang tìm dữ liệu tại: ../data/raw


Tải danh sách ảnh vào DataFrame

In [2]:
def load_image_paths(data_dir):
    image_dir = Path(data_dir)
    # Tìm tất cả các đuôi ảnh phổ biến
    filepaths = list(image_dir.glob(r'**/*.jpg')) + list(image_dir.glob(r'**/*.png')) + list(image_dir.glob(r'**/*.jpeg'))
    
    # Lấy tên thư mục cha làm nhãn (Label)
    labels = [os.path.split(os.path.split(filepath)[0])[1] for filepath in filepaths]

    filepaths = pd.Series(filepaths, name='Filepath').astype(str)
    labels = pd.Series(labels, name='Label')

    df = pd.concat([filepaths, labels], axis=1)
    
    # Trộn ngẫu nhiên (Shuffle)
    df = df.sample(frac=1).reset_index(drop=True)
    return df

# Chạy hàm tải dữ liệu
try:
    df = load_image_paths(DATA_DIR)
    print(f"Tổng số ảnh tìm thấy: {len(df)}")
    print("\nSố lượng ảnh mỗi lớp:")
    print(df['Label'].value_counts())
except Exception as e:
    print(f"Lỗi tìm ảnh: {e}")

Tổng số ảnh tìm thấy: 18

Số lượng ảnh mỗi lớp:
Label
pins_Messi       6
pins_Benzenma    6
pins_Ronaldo     6
Name: count, dtype: int64


Xây dựng kiến trúc Custom CNN

In [3]:
def build_custom_cnn(num_classes):
    model = Sequential()
    
    # --- BLOCK 1 ---
    # Input nhận ảnh 224x224x3
    model.add(Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3)))
    # Conv2D: Trích xuất đặc trưng
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
    # MaxPooling: Giảm kích thước ảnh đi một nửa
    model.add(MaxPooling2D((2, 2)))
    
    # --- BLOCK 2 ---
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    
    # --- BLOCK 3 ---
    model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    
    # --- PHẦN PHÂN LOẠI (CLASSIFIER) ---
    model.add(Flatten()) # Duỗi phẳng dữ liệu
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5)) # Tắt ngẫu nhiên 50% neuron để chống học vẹt (Overfitting)
    
    # Lớp đầu ra (Output Layer)
    model.add(Dense(num_classes, activation='softmax'))
    
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

print("Đã khởi tạo hàm build_custom_cnn thành công!")

Đã khởi tạo hàm build_custom_cnn thành công!


Huấn luyện K-Fold

In [4]:
# Khởi tạo K-Fold
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

# --- CẤU HÌNH DATA GENERATOR (Quan trọng: Rescale 1./255) ---
train_datagen = ImageDataGenerator(
    rescale=1./255,         # Chuẩn hóa pixel về [0, 1]
    rotation_range=20,      # Xoay ảnh ngẫu nhiên
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    rescale=1./255          # Chỉ chuẩn hóa, không xoay lật
)

acc_per_fold = []
loss_per_fold = []
fold_no = 1

# BẮT ĐẦU VÒNG LẶP
for train_index, val_index in kf.split(df):
    print(f"\nTraining Custom CNN for Fold {fold_no} ...")
    
    train_data = df.iloc[train_index]
    val_data = df.iloc[val_index]
    
    # 1. Tạo Train Generator
    train_gen = train_datagen.flow_from_dataframe(
        train_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    
    # Lấy danh sách lớp đầy đủ để tránh lỗi thiếu class
    full_classes = list(train_gen.class_indices.keys())
    
    # 2. Tạo Val Generator (Ép buộc dùng full_classes)
    val_gen = val_datagen.flow_from_dataframe(
        val_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=False,
        classes=full_classes 
    )
    
    # 3. Gọi hàm xây dựng Custom CNN
    num_classes = len(full_classes)
    model = build_custom_cnn(num_classes)
    
    # 4. Callbacks (Lưu file tên khác để không đè lên model VGG cũ)
    checkpoint_path = f"../models/custom_cnn_fold_{fold_no}.h5"
    if 'google.colab' in str(get_ipython()):
         checkpoint_path = f"/content/drive/MyDrive/models/custom_cnn_fold_{fold_no}.h5"

    callbacks = [
        ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ]
    
    # 5. Training
    try:
        history = model.fit(
            train_gen,
            epochs=EPOCHS, 
            validation_data=val_gen,
            callbacks=callbacks
        )
        
        # Ghi nhận kết quả
        scores = model.evaluate(val_gen, verbose=0)
        print(f'-> Kết quả Fold {fold_no}: Accuracy = {scores[1]*100:.2f}%')
        acc_per_fold.append(scores[1] * 100)
        loss_per_fold.append(scores[0])
        
    except Exception as e:
        print(f"Lỗi tại Fold {fold_no}: {e}")

    # Dọn dẹp RAM
    K.clear_session()
    fold_no += 1

# Báo cáo cuối cùng
print("\n" + "="*30)
if len(acc_per_fold) > 0:
    print(f"TRUNG BÌNH CỘNG (Custom CNN): {np.mean(acc_per_fold):.2f}%")
else:
    print("Chưa chạy xong fold nào.")
print("="*30)


Training Custom CNN for Fold 1 ...
Found 14 validated image filenames belonging to 3 classes.
Found 4 validated image filenames belonging to 3 classes.
Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2143 - loss: 1.1181
Epoch 1: val_accuracy improved from None to 0.50000, saving model to ../models/custom_cnn_fold_1.h5


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.2143 - loss: 1.1181 - val_accuracy: 0.5000 - val_loss: 3.6778
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3571 - loss: 2.9365
Epoch 2: val_accuracy did not improve from 0.50000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3571 - loss: 2.9365 - val_accuracy: 0.0000e+00 - val_loss: 2.5533
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1000ms/step - accuracy: 0.2143 - loss: 2.5546
Epoch 3: val_accuracy did not improve from 0.50000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.2143 - loss: 2.5546 - val_accuracy: 0.5000 - val_loss: 1.5189
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 913ms/step - accuracy: 0.3571 - loss: 1.4870
Epoch 4: val_accuracy did not improve from 0.50000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3571 - loss: 1.4870 - val_accuracy: 0.2500 - val_loss: 1.2636
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 922ms/step - accuracy: 0.2857 - loss: 1.2127
Epoch 5: val_accuracy did not improve from 0.50000
1/1 ━━━━━━


Training Custom CNN for Fold 2 ...
Found 14 validated image filenames belonging to 3 classes.
Found 4 validated image filenames belonging to 3 classes.
Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4286 - loss: 1.0882
Epoch 1: val_accuracy improved from None to 0.25000, saving model to ../models/custom_cnn_fold_2.h5


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.4286 - loss: 1.0882 - val_accuracy: 0.2500 - val_loss: 1.5786
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5000 - loss: 1.5066
Epoch 2: val_accuracy did not improve from 0.25000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 1.5066 - val_accuracy: 0.2500 - val_loss: 1.2011
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3571 - loss: 1.7753
Epoch 3: val_accuracy did not improve from 0.25000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3571 - loss: 1.7753 - val_accuracy: 0.2500 - val_loss: 1.5138
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 963ms/step - accuracy: 0.5000 - loss: 1.1575
Epoch 4: val_accuracy did not improve from 0.25000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 1.1575 - val_accuracy: 0.2500 - val_loss: 1.2443
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 955ms/step - accuracy: 0.4286 - loss: 1.0459
Epoch 5: val_accuracy improved from 0.25000 to 0.50000, saving model 

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4286 - loss: 1.0459 - val_accuracy: 0.5000 - val_loss: 1.1275
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 980ms/step - accuracy: 0.4286 - loss: 1.0948
Epoch 6: val_accuracy improved from 0.50000 to 0.75000, saving model to ../models/custom_cnn_fold_2.h5


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4286 - loss: 1.0948 - val_accuracy: 0.7500 - val_loss: 1.1371
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5714 - loss: 0.9898
Epoch 7: val_accuracy did not improve from 0.75000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5714 - loss: 0.9898 - val_accuracy: 0.7500 - val_loss: 1.1466
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 937ms/step - accuracy: 0.4286 - loss: 1.0883
Epoch 8: val_accuracy did not improve from 0.75000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4286 - loss: 1.0883 - val_accuracy: 0.5000 - val_loss: 1.1291
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 971ms/step - accuracy: 0.5000 - loss: 0.9646
Epoch 9: val_accuracy did not improve from 0.75000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 0.9646 - val_accuracy: 0.2500 - val_loss: 1.1216
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 969ms/step - accuracy: 0.7857 - loss: 0.8196
Epoch 10: val_accuracy did not improve from 0.75000
1/1 ━━━━━━━━━

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.3571 - loss: 1.0942 - val_accuracy: 0.0000e+00 - val_loss: 7.1870
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 842ms/step - accuracy: 0.3571 - loss: 2.5940
Epoch 2: val_accuracy improved from 0.00000 to 0.25000, saving model to ../models/custom_cnn_fold_3.h5


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3571 - loss: 2.5940 - val_accuracy: 0.2500 - val_loss: 5.0313
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 893ms/step - accuracy: 0.2857 - loss: 3.7882
Epoch 3: val_accuracy did not improve from 0.25000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.2857 - loss: 3.7882 - val_accuracy: 0.2500 - val_loss: 1.9038
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 876ms/step - accuracy: 0.3571 - loss: 1.6497
Epoch 4: val_accuracy did not improve from 0.25000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3571 - loss: 1.6497 - val_accuracy: 0.2500 - val_loss: 1.1576
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 901ms/step - accuracy: 0.5000 - loss: 0.9250
Epoch 5: val_accuracy did not improve from 0.25000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 0.9250 - val_accuracy: 0.0000e+00 - val_loss: 1.3152
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 889ms/step - accuracy: 0.5714 - loss: 1.0955
Epoch 6: val_accuracy did not improve from 0.25000
1/1 ━━━━

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.4000 - loss: 1.1043 - val_accuracy: 0.3333 - val_loss: 1.5396
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2667 - loss: 2.2207
Epoch 2: val_accuracy did not improve from 0.33333
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.2667 - loss: 2.2207 - val_accuracy: 0.3333 - val_loss: 1.5807
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4667 - loss: 1.3097
Epoch 3: val_accuracy did not improve from 0.33333
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4667 - loss: 1.3097 - val_accuracy: 0.3333 - val_loss: 1.3834
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2000 - loss: 1.5352
Epoch 4: val_accuracy did not improve from 0.33333
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.2000 - loss: 1.5352 - val_accuracy: 0.0000e+00 - val_loss: 1.1827
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2667 - loss: 1.1670
Epoch 5: val_accuracy did not improve from 0.33333
1/1 ━━━━━━━━━━━━━━━━

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.4667 - loss: 1.0879 - val_accuracy: 0.3333 - val_loss: 4.4081
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 981ms/step - accuracy: 0.3333 - loss: 4.0821
Epoch 2: val_accuracy did not improve from 0.33333
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3333 - loss: 4.0821 - val_accuracy: 0.3333 - val_loss: 2.9553
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 983ms/step - accuracy: 0.3333 - loss: 4.6392
Epoch 3: val_accuracy did not improve from 0.33333
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3333 - loss: 4.6392 - val_accuracy: 0.3333 - val_loss: 1.3538
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 948ms/step - accuracy: 0.4000 - loss: 1.6550
Epoch 4: val_accuracy did not improve from 0.33333
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4000 - loss: 1.6550 - val_accuracy: 0.3333 - val_loss: 1.2444
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 956ms/step - accuracy: 0.4000 - loss: 1.2828
Epoch 5: val_accuracy did not improve from 0.33333
1/1 ━━━━━━━━